In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib

from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_DIR = Path(
    r"F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand"
)

DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
FORECAST_DIR = OUTPUT_DIR / "forecasts"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths configured successfully!")
print("Project:", PROJECT_DIR)

Project paths configured successfully!
Project: F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand


In [3]:
from pathlib import Path
import pandas as pd

# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_ROOT = Path(
    r"F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

# Create outputs folder if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# PREDICTION FILE
# ============================================================

PREDICTION_PATH = OUTPUT_DIR / "model_training_predictions.csv"

print("Looking for prediction file:")
print(PREDICTION_PATH)
print()

# ============================================================
# CHECK FILE
# ============================================================

if not PREDICTION_PATH.exists():
    print("❌ Prediction file not found!")
    print()
    print("Expected location:")
    print(PREDICTION_PATH)
    print()
    print("Please run 06_model_training.ipynb first.")
else:
    # ========================================================
    # LOAD PREDICTIONS
    # ========================================================

    predictions = pd.read_csv(PREDICTION_PATH)

    print("✅ Predictions loaded successfully!")
    print("Shape:", predictions.shape)
    print()
    print("Columns:")
    print(predictions.columns.tolist())
    print()
    
    display(predictions.head())

Looking for prediction file:
F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\outputs\model_training_predictions.csv

❌ Prediction file not found!

Expected location:
F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\outputs\model_training_predictions.csv

Please run 06_model_training.ipynb first.


In [6]:
# ============================================================
# 06 - MODEL TRAINING
# AI-DRIVEN LOGISTICS BOOKING FORECASTING
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand"
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)


# ============================================================
# 2. LOAD PROCESSED DATA
# ============================================================

DATA_PATH = PROCESSED_DIR / "daily_booking_demand.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"\nProcessed dataset not found:\n{DATA_PATH}\n\n"
        "Please run 02_data_cleaning.ipynb first."
    )

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


# ============================================================
# 3. IDENTIFY DATE AND TARGET COLUMNS
# ============================================================

date_col = "DATE"
demand_col = "BOOKING_COUNT"

if date_col not in df.columns:
    raise ValueError(
        f"Date column '{date_col}' not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )

if demand_col not in df.columns:
    raise ValueError(
        f"Booking demand column '{demand_col}' not found.\n"
        f"Available columns: {df.columns.tolist()}"
    )


# ============================================================
# 4. PREPARE DATA
# ============================================================

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[demand_col] = pd.to_numeric(
    df[demand_col],
    errors="coerce"
)

df = df.dropna(
    subset=[date_col, demand_col]
)

df = df.sort_values(date_col).reset_index(drop=True)

print("\nPrepared dataset:")
print("Shape:", df.shape)
print("Date range:")
print(df[date_col].min(), "to", df[date_col].max())


# ============================================================
# 5. FEATURE ENGINEERING
# ============================================================

df["year"] = df[date_col].dt.year
df["month"] = df[date_col].dt.month
df["day"] = df[date_col].dt.day
df["day_of_week"] = df[date_col].dt.dayofweek
df["week_of_year"] = df[date_col].dt.isocalendar().week.astype(int)
df["quarter"] = df[date_col].dt.quarter

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

df["is_month_start"] = (
    df[date_col].dt.is_month_start
).astype(int)

df["is_month_end"] = (
    df[date_col].dt.is_month_end
).astype(int)


# ============================================================
# LAG FEATURES
# ============================================================

df["lag_1"] = df[demand_col].shift(1)
df["lag_2"] = df[demand_col].shift(2)
df["lag_3"] = df[demand_col].shift(3)
df["lag_7"] = df[demand_col].shift(7)
df["lag_14"] = df[demand_col].shift(14)
df["lag_28"] = df[demand_col].shift(28)


# ============================================================
# ROLLING FEATURES
# ============================================================

df["rolling_mean_3"] = (
    df[demand_col]
    .shift(1)
    .rolling(3)
    .mean()
)

df["rolling_mean_7"] = (
    df[demand_col]
    .shift(1)
    .rolling(7)
    .mean()
)

df["rolling_mean_14"] = (
    df[demand_col]
    .shift(1)
    .rolling(14)
    .mean()
)

df["rolling_std_7"] = (
    df[demand_col]
    .shift(1)
    .rolling(7)
    .std()
)


# ============================================================
# CHANGE FEATURES
# ============================================================

df["demand_change_1"] = (
    df[demand_col].shift(1)
    - df[demand_col].shift(2)
)

df["demand_change_7"] = (
    df[demand_col].shift(1)
    - df[demand_col].shift(8)
)


# ============================================================
# CYCLICAL FEATURES
# ============================================================

df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)


# ============================================================
# REMOVE MISSING VALUES
# ============================================================

df = df.dropna().reset_index(drop=True)

print("\nAfter feature engineering:")
print("Shape:", df.shape)


# ============================================================
# 6. DEFINE FEATURES
# ============================================================

feature_columns = [
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_3",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_std_7",
    "demand_change_1",
    "demand_change_7",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos"
]

X = df[feature_columns]
y = df[demand_col]


# ============================================================
# 7. TIME-SERIES TRAIN / TEST SPLIT
# ============================================================

# 80% training, 20% testing
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

test_dates = df[date_col].iloc[split_index:].copy()

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining period:")
print(
    df[date_col].iloc[0],
    "to",
    df[date_col].iloc[split_index - 1]
)

print("\nTesting period:")
print(
    df[date_col].iloc[split_index],
    "to",
    df[date_col].iloc[-1]
)


# ============================================================
# 8. DEFINE MODELS
# ============================================================

models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}


# ============================================================
# 9. TRAIN MODELS
# ============================================================

trained_models = {}

results = []

print("\n" + "=" * 60)
print("MODEL TRAINING")
print("=" * 60)

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    model.fit(X_train, y_train)

    trained_models[model_name] = model

    predictions_temp = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions_temp
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions_temp
        )
    )

    r2 = r2_score(
        y_test,
        predictions_temp
    )

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    print("Training completed.")
    print("MAE :", round(mae, 2))
    print("RMSE:", round(rmse, 2))
    print("R²  :", round(r2, 4))


# ============================================================
# 10. MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame(results)

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

display(comparison)


# ============================================================
# 11. CREATE PREDICTION DATAFRAME
# ============================================================

prediction_results = pd.DataFrame({
    "DATE": test_dates.values,
    "Actual": y_test.values
})

for model_name, model in trained_models.items():

    prediction_results[model_name] = model.predict(
        X_test
    )


# ============================================================
# 12. SAVE PREDICTIONS
# ============================================================

PREDICTION_PATH = (
    OUTPUT_DIR / "model_training_predictions.csv"
)

prediction_results.to_csv(
    PREDICTION_PATH,
    index=False
)

print("\n" + "=" * 60)
print("PREDICTIONS SAVED")
print("=" * 60)

print("File:")
print(PREDICTION_PATH)

print("\nShape:")
print(prediction_results.shape)

print("\nColumns:")
print(prediction_results.columns.tolist())

display(prediction_results.head())


# ============================================================
# 13. SAVE MODEL COMPARISON
# ============================================================

COMPARISON_PATH = (
    OUTPUT_DIR / "model_comparison.csv"
)

comparison.to_csv(
    COMPARISON_PATH,
    index=False
)

print("\nModel comparison saved:")
print(COMPARISON_PATH)


# ============================================================
# 14. FINAL CHECK
# ============================================================

print("\n" + "=" * 60)
print("FINAL CHECK")
print("=" * 60)

print("Prediction file exists:",
      PREDICTION_PATH.exists())

print("Comparison file exists:",
      COMPARISON_PATH.exists())

print("\nTraining completed successfully! ✅")

Project root:
F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand

Dataset loaded successfully!
Shape: (31, 2)
Columns: ['DATE', 'BOOKING_COUNT']


,DATE,BOOKING_COUNT
0,2026-08-01,53116
1,2026-08-02,7336
2,2026-08-03,46339
3,2026-08-04,49664
4,2026-08-05,48974



Prepared dataset:
Shape: (31, 2)
Date range:
2026-08-01 00:00:00 to 2026-08-31 00:00:00

After feature engineering:
Shape: (3, 27)

TRAIN / TEST SPLIT
Training samples: 2
Testing samples : 1

Training period:
2026-08-29 00:00:00 to 2026-08-30 00:00:00

Testing period:
2026-08-31 00:00:00 to 2026-08-31 00:00:00

MODEL TRAINING

Training Random Forest...
Training completed.
MAE : 22879.88
RMSE: 22879.88
R²  : nan

Training Decision Tree...
Training completed.
MAE : 1176.0
RMSE: 1176.0
R²  : nan

Training Gradient Boosting...
Training completed.
MAE : 23430.39
RMSE: 23430.39
R²  : nan

MODEL COMPARISON


,Model,MAE,RMSE,R2
0,Random Forest,22879.875000,22879.875000,NaN
1,Decision Tree,1176.000000,1176.000000,NaN
2,Gradient Boosting,23430.386697,23430.386697,NaN



PREDICTIONS SAVED
File:
F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\outputs\model_training_predictions.csv

Shape:
(1, 5)

Columns:
['DATE', 'Actual', 'Random Forest', 'Decision Tree', 'Gradient Boosting']


,DATE,Actual,Random Forest,Decision Tree,Gradient Boosting
0,2026-08-31,55187,32307.125,54011.0,31756.613303



Model comparison saved:
F:\AI-Driven Time-Series Forecasting and Predictive Analytics for Logistics Booking Demand\outputs\model_comparison.csv

FINAL CHECK
Prediction file exists: True
Comparison file exists: True

Training completed successfully! ✅


In [5]:
print("Number of rows:", len(df_raw))

for column in df_raw.columns:
    print(
        f"{column}: "
        f"{df_raw[column].notna().sum()} non-null values"
    )

NameError: name 'df_raw' is not defined

In [7]:
date_col = next(
    (c for c in predictions.columns
     if c.lower() in ["date", "bkdate", "booking_date"]),
    None
)

actual_col = next(
    (c for c in predictions.columns
     if c.lower() in ["actual", "actual_demand", "y_test", "demand"]),
    None
)

if date_col is None:
    raise ValueError("Date column not found in prediction file.")

if actual_col is None:
    raise ValueError("Actual demand column not found in prediction file.")

predictions[date_col] = pd.to_datetime(
    predictions[date_col],
    errors="coerce"
)

predictions[actual_col] = pd.to_numeric(
    predictions[actual_col],
    errors="coerce"
)

predictions = predictions.dropna(
    subset=[date_col, actual_col]
).sort_values(date_col).reset_index(drop=True)

print("Date column:", date_col)
print("Actual demand column:", actual_col)
print("Rows available:", len(predictions))

NameError: name 'predictions' is not defined

In [8]:
model_names = [
    "Linear Regression",
    "Decision Tree",
    "Random Forest",
    "Gradient Boosting"
]

prediction_columns = {}

for model_name in model_names:
    possible_names = [
        model_name,
        model_name.replace(" ", "_"),
        model_name.lower().replace(" ", "_")
    ]

    for col in predictions.columns:
        if col in possible_names:
            prediction_columns[model_name] = col
            break

print("Detected model prediction columns:")

for model_name, col in prediction_columns.items():
    print(f"{model_name}: {col}")

if not prediction_columns:
    raise ValueError(
        "No model prediction columns were found. "
        "Check the columns in model_training_predictions.csv."
    )

NameError: name 'predictions' is not defined

In [9]:
for col in prediction_columns.values():
    predictions[col] = pd.to_numeric(
        predictions[col],
        errors="coerce"
    )

required_cols = [
    actual_col,
    *prediction_columns.values()
]

evaluation_data = predictions.dropna(
    subset=required_cols
).copy()

print("Rows before cleaning:", len(predictions))
print("Rows after cleaning:", len(evaluation_data))

display(evaluation_data.head())

NameError: name 'actual_col' is not defined